# All Weather — Replication and Evaluation

**Universe (core 5):** `SPY`, `TLT`, `IEF`, `GLD`, `DBC`  
**Data file:** `FINA4359 database +btc.csv` (teammate source; BTC/SVXY excluded for core AWP).  
**Thesis:** Replace the *Macro* section with your exact macro series when available. Placeholders are labeled `DEFAULT` only.


In [1]:
import warnings
import io
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import statsmodels.api as sm
except ImportError:
    sm = None

try:
    import yfinance as yf
except ImportError:
    yf = None

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib as mpl
    mpl.use("Agg")
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(0)
ANN = 252
ORDER = ["spy", "tlt", "ief", "gld", "dbc"]
# 30% stocks, 40% long Treas, 15% interm, 7.5% gold, 7.5% commo (teammate / Dalio template)
W_FIXED = pd.Series([0.30, 0.40, 0.15, 0.075, 0.075], index=ORDER)


In [2]:
# Load prices (teammate column mapping)
raw = pd.read_csv("FINA4359 database +btc.csv")
raw.columns = ["time", "btc", "dbc", "gld", "ief", "spy", "svxy", "tlt"]
raw["time"] = pd.to_datetime(raw["time"], dayfirst=True, errors="coerce")
prices = raw.set_index("time").sort_index()[ORDER].copy()
prices = prices.loc[prices.notna().all(axis=1)].astype(float)
ret = prices.pct_change().dropna(how="any")
r = ret[ORDER]
print("From", ret.index[0].date(), "to", ret.index[-1].date(), "obs", len(ret))


From 2014-09-18 to 2026-04-17 obs 2912


In [65]:
# Portfolio helpers

def backtest_const(r: pd.DataFrame, w: pd.Series) -> pd.Series:
    w = w.reindex(r.columns).fillna(0.0)
    return (r * w).sum(axis=1)

def backtest_invvol_static(r: pd.DataFrame) -> pd.Series:
    s = r.std() * np.sqrt(ANN)
    w = (1.0 / s) / (1.0 / s).sum()
    return (r * w.reindex(r.columns).fillna(0.0)).sum(axis=1)

def backtest_invvol_rebal(r: pd.DataFrame, lookback: int, rule: str) -> pd.Series:
    # rule: "M" = month-end, "6M" = semi-annual, "Y" = year-end
    vol = r.rolling(lookback).std() * np.sqrt(ANN)
    invv = 1.0 / vol.replace(0, np.nan)
    w_raw = invv.div(invv.sum(axis=1), axis=0)
    fmap = {"M": "ME", "6M": "6M", "Y": "YE"}
    f = fmap[rule]
    w_rb = w_raw.resample(f).last().reindex(r.index, method="ffill")
    w = w_rb.shift(1)  # trade next day
    return (r * w).sum(axis=1)

def max_dd(s: pd.Series) -> float:
    c = (1.0 + s).cumprod()
    pk = c.cummax()
    return float((c / pk - 1.0).min())

def stats(s: pd.Series, n: str = ""):
    s = s.dropna()
    nobs = len(s)
    cagr = (1.0 + s).prod() ** (ANN / nobs) - 1.0
    vol = s.std() * np.sqrt(ANN)
    sh = (s.mean() * ANN) / vol if vol > 0 else np.nan
    dstd = s[s < 0].std() * np.sqrt(ANN) if (s < 0).any() else np.nan
    sort = (s.mean() * ANN) / dstd if dstd and dstd > 0 else np.nan
    return {
        "name": n, "CAGR": cagr, "ann_vol": vol, "Sharpe": sh,
        "Sortino": sort, "maxDD": max_dd(s),
    }
def stats_mut(s: pd.Series, n: str = ""):
    s = s.dropna()
    nobs = len(s)
    totret = (1.0 + s).prod() - 1.0
    cagr = (1.0 + s).prod() ** (ANN / nobs) - 1.0
    vol = s.std() * np.sqrt(ANN)
    sh = cagr / vol if vol > 0 else np.nan
    dstd = s[s < 0].std() * np.sqrt(ANN) if (s < 0).any() else np.nan
    sort = (s.mean() * ANN) / dstd if dstd and dstd > 0 else np.nan
    return {
        'Total Return (%)': totret, "CAGR (%)": cagr, "Vol (%)": vol, "Sharpe": sh,
        "MDD (%)": max_dd(s),
    }


## 1) Replication: fixed + inverse-vol + 60/40 benchmark


In [4]:
# --- Replication
port_fixed = backtest_const(r, W_FIXED)
w_6040 = pd.Series({"spy": 0.60, "tlt": 0.20, "ief": 0.20, "gld": 0.0, "dbc": 0.0})
port_6040 = backtest_const(r, w_6040)
port_iv_s = backtest_invvol_static(r)
port_iv_m = backtest_invvol_rebal(r, 252, "M")
port_iv_6m = backtest_invvol_rebal(r, 252, "6M")
port_iv_y = backtest_invvol_rebal(r, 252, "Y")

# Static inverse-vol weights (one number per asset) — for comparison / display
s_ann = r.std() * np.sqrt(ANN)
w_iv_stat = (1.0 / s_ann) / (1.0 / s_ann).sum()
print("Inverse-vol (full-sample) weights:")
print(w_iv_stat.round(4))


Inverse-vol (full-sample) weights:
spy    0.1436
tlt    0.1702
ief    0.3852
gld    0.1591
dbc    0.1419
dtype: float64


In [21]:
r.std()*16

spy    0.177644
tlt    0.149819
ief    0.066219
gld    0.160342
dbc    0.179738
dtype: float64

In [52]:
rename_dict = {
    'spy': 'SPY – stocks',
    'gld': 'GLD – gold',
    'dbc': 'DBC – commodities',
    'ief': 'IEF – intermediate bonds',
    'tlt': 'TLT – long term bonds'
}
tbl = pd.DataFrame(
    {
        'Full Hist. Annual SD (%)': r.std()*252**0.5,
        'Portfolio Weight (%)': (1/r.std())/(1/r.std()).sum()
    }
).loc[rename_dict.keys()]
tbl['Assigned Vol (%)'] = tbl['Full Hist. Annual SD (%)'] * tbl['Portfolio Weight (%)']
tbl.rename(rename_dict).applymap(lambda x: f"{x:.2%}")

,Full Hist. Annual SD (%),Portfolio Weight (%),Assigned Vol (%)
SPY – stocks,17.63%,14.36%,2.53%
GLD – gold,15.91%,15.91%,2.53%
DBC – commodities,17.83%,14.19%,2.53%
IEF – intermediate bonds,6.57%,38.52%,2.53%
TLT – long term bonds,14.86%,17.02%,2.53%


## 2) Performance table & growth of $1


In [66]:
strat = pd.DataFrame(
    {
        "AWP_fixed_30/40/15/7.5/7.5": port_fixed,
        "AWP_invvol_static": port_iv_s,
        "AWP_invvol_monthly": port_iv_m,
        "AWP_invvol_6M": port_iv_6m,
        "AWP_invvol_annual": port_iv_y,
        "bench_60-40 (SPY60_IEF/TLT20-20)": port_6040,
    }
)
pd.DataFrame([stats(strat[k], k) for k in strat])
df = pd.DataFrame({
    'Market (S&P 500)': pd.Series(stats_mut(r['spy'],'mkt')),
    'AWP': pd.Series(stats_mut(port_iv_s,'AWP'))
})
for col in df.index:
    if "Sharpe" in col:
        df.loc[col] = df.loc[col].apply(lambda x: f"{x:.2f}")  # keep as .2% for consistency, albeit usually would be ratio/decimal
    else:
        df.loc[col] = df.loc[col].apply(lambda x: f"{x:.2%}")
df

,Market (S&P 500),AWP
Total Return (%),331.52%,87.42%
CAGR (%),13.49%,5.59%
Vol (%),17.63%,7.15%
Sharpe,0.77,0.78
MDD (%),-33.72%,-17.56%


In [ ]:
# Performance — cumulative
cum = (1.0 + strat).cumprod()
ax = cum.plot(figsize=(11, 5), title="Growth of $1 (daily, lagged rebal for dynamic)")
ax.set_ylabel("Wealth")
plt.tight_layout()
plt.show()


In [ ]:
# Rolling 12M — pick main strategy
sel = "AWP_fixed_30/40/15/7.5/7.5"
s = strat[sel]
w = 252
r12 = (1.0 + s).rolling(w).apply(lambda x: x.prod() - 1, raw=True)
v12 = s.rolling(w).std() * np.sqrt(ANN)
m12 = s.rolling(w).mean() * ANN
sh12 = m12 / v12
fig, axs = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
axs[0].plot(r12, color="C0")
axs[0].set_ylabel("Rolling 12M ret")
axs[1].plot(v12, color="C1")
axs[1].set_ylabel("Ann. vol 12M")
axs[2].plot(sh12, color="C2")
axs[2].set_ylabel("Ann.ret/vol 12M")
fig.suptitle("Rolling: " + sel)
plt.tight_layout()
plt.show()


In [ ]:
# Subsample Sharpe (mean/vol) for comparison
cuts = {
    "to2019": (strat.index[0], "2019-12-31"),
    "2020-21": ("2020-01-01", "2021-12-31"),
    "2022": ("2022-01-01", "2022-12-31"),
    "2023p": ("2023-01-01", strat.index[-1]),
}
row = []
for wn, (a, b) in cuts.items():
    sub = strat.loc[a:b]
    for col in sub.columns:
        t = sub[col]
        mu, vo = t.mean() * ANN, t.std() * np.sqrt(ANN)
        row.append((wn, col, mu / vo if vo else np.nan, (1 + t).prod() ** (ANN / len(t)) - 1, vo))
sp = pd.DataFrame(row, columns=["window", "strat", "Sharpe", "CAGR", "ann_vol"])
sp.pivot_table(index="strat", columns="window", values="Sharpe")


## 3) PnL / risk (fixed) + 60/40 in table above; FF5 below


In [ ]:
# PnL attribution — arithmetic, AWP fixed
contrib = r.mul(W_FIXED, axis=1)
cc = (1.0 + contrib).cumprod()
cc.plot(figsize=(10, 4), title="Cumulative (1+r) by sleeve, fixed AWP")
plt.legend(loc="upper left", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
# Simple share of summed daily contributions
sdi = contrib.sum()
print("Share of summed daily PnL contributions:\n", (sdi / sdi.sum()).round(4))


In [ ]:
# Marginal + component vol (ann.) — fixed weights
S = (r[ORDER].cov() * ANN)
w0 = W_FIXED.reindex(ORDER).values
sig_p = float(np.sqrt(w0 @ S.values @ w0))
mrc = S.values @ w0
crc = w0 * mrc / sig_p
pd.DataFrame({"% vol": 100.0 * crc / sig_p, "w": w0}, index=ORDER)


In [ ]:
# Tail: 5% Expected Shortfall (daily) — portfolio vs sum of components on tail days
pr = (r * W_FIXED).sum(axis=1)
q5 = pr.quantile(0.05)
mask = pr <= q5
es_p = pr[mask].mean()
# Mean sleeve return on the same days (arithmetic, indicative)
es_each = (r[mask] * W_FIXED).mean()
print("5% ES portfolio (daily return):", es_p)
print("Component mean on tail days (not additive ES):")
print(es_each)


## 4) Fama–French 5 (time series on AWP excess)


In [ ]:
# --- FF5: Kenneth French daily 5 + RF

def load_ff5():
    u = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily.CSV"
    with urlopen(u) as resp:
        t = resp.read().decode("utf-8", errors="replace")
    lines = t.splitlines()
    st = next(i for i, ln in enumerate(lines) if ln.startswith("Mkt-RF") or ("," in ln and "Mkt-RF" in ln))
    dfl = pd.read_csv(io.StringIO("\n".join(lines[st:])), nrows=300_000)
    dfl = dfl.rename(columns={dfl.columns[0]: "date"})
    dfl["date"] = pd.to_datetime(dfl["date"].astype(str), format="%Y%m%d", errors="coerce")
    dfl = dfl.set_index("date")
    for col in dfl.columns:
        dfl[col] = pd.to_numeric(dfl[col], errors="coerce")
    dfl = dfl / 100.0
    return dfl.dropna(how="all")

ff = load_ff5()
j = pd.DataFrame({"port": port_fixed, "RF": ff["RF"]}).join(ff[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], how="inner")
j["Y"] = j["port"] - j["RF"]
Xf = j[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]]
if sm is not None:
    res = sm.OLS(j["Y"], sm.add_constant(Xf), missing="drop").fit()
    print(res.summary().tables[1])
    print("Adj.R2 =", res.rsquared_adj, "alpha =", res.params[0] * ANN, "(daily->Ann approx *252)")
else:
    Xv = sm.add_constant(Xf) if sm else np.column_stack([np.ones(len(j)), Xf.values])
    b, _, _, _ = np.linalg.lstsq(Xv, j["Y"].values, rcond=None)
    print("betas const + FF5", b)


In [ ]:
# Rolling FF5: R2 and mkt loading (36M = 252*3? use 3*252 = 756 days)
J = j.dropna()
win = 504  # 24M trading
if sm is not None and len(J) > win + 5:
    out = []
    yv = J["Y"].values
    Xa = J[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]].values
    c = len(J)
    for i in range(win, c):
        yi = yv[i - win : i]
        Xi = sm.add_constant(Xa[i - win : i, :])
        reg = sm.OLS(yi, Xi, missing="drop").fit()
        out.append((J.index[i], reg.rsquared_adj, reg.params[0], reg.params[1]))
    rollf = pd.DataFrame(out, columns=["d", "R2a", "alpha_d", "mkt_d"]).set_index("d")
    fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    ax[0].plot(rollf["R2a"], color="C0")
    ax[0].set_ylabel("Adj.R2")
    ax[1].plot(rollf["mkt_d"], color="C1")
    ax[1].set_ylabel("Mkt factor loading")
    plt.suptitle("Rolling 24M OLS: FF5 on AWP excess")
    plt.tight_layout()
    plt.show()


## Diversification / protection (asset-level)


In [ ]:
# Average pairwise rolling correlation (5 assets)
apc = []
for i in range(252, len(r)):
    sub = r.iloc[i - 252 : i]
    C = sub.corr().values
    u = np.triu_indices_from(C, 1)
    apc.append((r.index[i], C[u].mean()))
apc = pd.DataFrame(apc, columns=["d", "avg_pair"] ).set_index("d")
apc["avg_pair"].plot(figsize=(10, 3), title="252d avg pairwise correlation")
plt.tight_layout()
plt.show()
print("HHI fixed AWP", float((W_FIXED**2).sum()))


In [ ]:
# PCA: PC1 share of trace(cov) rolling 252d
from numpy.linalg import eigh
pc1v = []
for i in range(252, len(r)):
    sub = r.iloc[i - 252 : i]
    C = sub.cov().values
    ev, _ = eigh(C)
    pc1v.append((r.index[i], ev.max() / ev.sum()))
pc1s = pd.DataFrame(pc1v, columns=["d", "s"]).set_index("d")
pc1s["s"].plot(figsize=(10, 3), title="PC1 / trace(Σ) rolling 252d")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
# Co-move: mean asset return in worst SPY *months* (bottom decile)
rm = r["spy"].resample("ME").sum()
q = rm.quantile(0.1)
bad = set(rm[rm <= q].index.to_period("M"))
mper = r.index.to_period("M")
sub = r.loc[[p in bad for p in mper]]
print("Bottom-decile SPY months, mean daily return by asset:")
print(sub[ORDER].mean().round(5))


## Macro: DEFAULT (replace with thesis)


In [ ]:
# Phase 1 (DEFAULT) — not your thesis. Merge your indicators into `Mthesis`.
def grab_yf(symbols, idx):
    if yf is None:
        return pd.DataFrame(index=idx)
    out = {}
    for sym in symbols:
        d = yf.download(sym, start=idx[0], end=idx[-1] + pd.Timedelta(1, "D"), auto_adjust=True, progress=False)
        if d.empty:
            continue
        s = d["Close"] if "Close" in d.columns else d.iloc[:, 0]
        s = pd.Series(np.asarray(s).ravel(), index=pd.to_datetime(d.index).tz_localize(None))
        out[sym] = s.reindex(idx, method="ffill")
    return pd.DataFrame(out, index=idx)

macro = grab_yf(["^TNX", "^VIX", "UUP"], r.index).ffill().bfill()
Mdef = pd.DataFrame(index=r.index)
Mdef["d_TNX"] = macro["^TNX"].diff()
Mdef["VIX_l"] = macro["^VIX"]
Mdef["d_UUP"] = macro["UUP"].pct_change()
Mdef["spy_63d"] = r["spy"].rolling(63).sum()
Mdef["dbc_63d"] = r["dbc"].rolling(63).sum()
# Phase 1 OLS: port ~ macro
D = pd.DataFrame({"y": port_fixed}).join(Mdef, how="inner").dropna()
if sm is not None and len(D) > 200:
    X = sm.add_constant(D.drop(columns=["y"]))
    print(sm.OLS(D["y"], X).fit().summary().tables[1])


In [ ]:
# Phase 2: augmented (VIX^2); compare adj. R2 vs smaller spec
D2 = D.copy()
D2["VIX2"] = D2["VIX_l"] ** 2
if sm is not None:
    X1 = sm.add_constant(D2.drop(columns=["y"]))
    r1 = sm.OLS(D2["y"], X1).fit()
    base = D2[["d_TNX", "VIX_l", "spy_63d", "dbc_63d"]]
    r0 = sm.OLS(D2["y"], sm.add_constant(base)).fit()
    print("AdjR2 full (incl. VIX2 d_UUP)", r1.rsquared_adj, "vs base", r0.rsquared_adj)


In [ ]:
# Rolling 504d: thesis macro (DEFAULT columns) on y = port
if sm is not None and len(D) > 600:
    y = D["y"]
    X0 = D.drop(columns=["y"])
    win, out = 504, []
    Xc = sm.add_constant(X0)
    for k in range(win, len(D)):
        reg = sm.OLS(y.iloc[k - win : k], Xc.iloc[k - win : k]).fit()
        out.append((D.index[k], reg.rsquared_adj))
    pd.DataFrame(out, columns=["d", "R2a"]).set_index("d")["R2a"].plot(figsize=(9, 3), title="Rolling 2Y macro (DEFAULT) R2")
    plt.tight_layout()
    plt.show()


## Thesis falsification / robustness


In [ ]:
# Start/end sensitivity: drop first/last 252d
p = port_fixed
core = p.iloc[252:-252] if len(p) > 504 else p
print("CAGR full", stats(p)["CAGR"], "trim", stats(core)["CAGR"])


In [ ]:
# Lookback 126/252/504 and monthly rebal
for L in [126, 252, 504]:
    s = backtest_invvol_rebal(r, L, "M")
    print("lookback", L, "CAGR~", stats(s)["CAGR"], "ann_vol", stats(s)["ann_vol"])


In [ ]:
# Transaction cost: 5bp per rebalance only on inv-vol dynamic (monthly)
# approximate: each month -0.0005 on rebal day (sum w change)
# simplified: -5bp*12 annual drag on monthly
print("Rule-of-thumb: ~6bp/yr for 5bp x 12 monthly round-turn on bonds (illustrative)")


In [ ]:
# 2022 bond-stocks: rolling 63d SPY-TLT correlation
ct = r["spy"].rolling(63).corr(r["tlt"])
ct.loc["2020":"2023"].plot(figsize=(9, 3), title="63d SPY-TLT correlation (2020-23)")
plt.axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()


## Short conclusion

- Compare **CAGR, vol, max DD, rolling FF5 R2** against **60/40** and **FF5 loadings**.
- If **PC1** share is high and **SPY-TLT** correlation is positive in stress, diversification claims are weaker in that period.
- **Replace DEFAULT macro** with thesis indicators, then re-run OLS/rolling R2. **Delete** `build_awp_nb.py` after this notebook is generated if present.
